# PreciseFlex 400 — Teaching & Freedrive Notebook

Uses the `JointArm` frontend with a `PreciseFlex400Backend`.

In [ ]:
import json
from pathlib import Path

from pylabrobot.arms.joint_arm import JointArm
from pylabrobot.brooks.precise_flex import (
  ElbowOrientation,
  PFAxis,
  PreciseFlex400Backend,
  PreciseFlexBackend,
)
from pylabrobot.resources import Coordinate, Resource, Rotation

POSITIONS_FILE = Path("positions.json")

In [ ]:
def save_position(name: str, pos):
  """Save a named position to disk."""
  positions = json.loads(POSITIONS_FILE.read_text()) if POSITIONS_FILE.exists() else {}
  positions[name] = {
    "x": pos.location.x,
    "y": pos.location.y,
    "z": pos.location.z,
    "roll": pos.rotation.x,
    "pitch": pos.rotation.y,
    "yaw": pos.rotation.z,
    "orientation": pos.orientation.value if pos.orientation else None,
  }
  POSITIONS_FILE.write_text(json.dumps(positions, indent=2))
  print(f"Saved '{name}'")


def load_position(name: str):
  """Load a named position from disk."""
  positions = json.loads(POSITIONS_FILE.read_text())
  p = positions[name]
  return (
    Coordinate(p["x"], p["y"], p["z"]),
    Rotation(p["roll"], p["pitch"], p["yaw"]),
    ElbowOrientation(p["orientation"]) if p["orientation"] else None,
  )


def list_positions():
  """List all saved positions."""
  if not POSITIONS_FILE.exists():
    print("No saved positions.")
    return
  positions = json.loads(POSITIONS_FILE.read_text())
  for name, p in positions.items():
    print(
      f"  {name}: x={p['x']:.1f} y={p['y']:.1f} z={p['z']:.1f} "
      f"yaw={p['yaw']:.1f} {p['orientation'] or ''}"
    )

## Connect

In [ ]:
backend = PreciseFlex400Backend(host="192.168.0.1")
reference = Resource("workcell", size_x=2000, size_y=2000, size_z=0)
arm = JointArm(backend=backend, reference_resource=reference)

await arm.setup()
print(f"Version: {await backend.request_version()}")

## Read current position

In [ ]:
cart = await arm.request_gripper_location()
print(f"Cartesian: x={cart.location.x:.1f}, y={cart.location.y:.1f}, z={cart.location.z:.1f}")
print(
  f"Rotation:  yaw={cart.rotation.z:.1f}, pitch={cart.rotation.y:.1f}, roll={cart.rotation.x:.1f}"
)
print(f"Elbow:     {cart.orientation}")
print()
joints = await arm.request_joint_position()
print(f"Joints: {joints}")

## Freedrive mode

Enter freedrive to manually position the arm. Use `[0]` to free all axes.

In [ ]:
await backend.start_freedrive_mode([0])
print("Freedrive ON -- move the arm manually")

In [ ]:
# Read position while in freedrive
cart = await arm.request_gripper_location()
print(
  f"x={cart.location.x:.1f}, y={cart.location.y:.1f}, z={cart.location.z:.1f}, "
  f"yaw={cart.rotation.z:.1f}, orientation={cart.orientation}"
)

In [ ]:
await backend.stop_freedrive_mode()
print("Freedrive OFF")

## Teach & save positions

Use freedrive to move the arm, then save the position to disk.

In [ ]:
# Save current position with a name
pos = await arm.request_gripper_location()
save_position("home", pos)

In [ ]:
# Save another position
pos = await arm.request_gripper_location()
save_position("plate_pickup", pos)

In [ ]:
list_positions()

## Replay saved positions

In [ ]:
loc, rot, orientation = load_position("home")
await arm.move_to_location(
  location=loc,
  direction=rot,
  backend_params=PreciseFlexBackend.MoveToLocationParams(orientation=orientation, speed=30),
)
print("Moved to 'home'")

In [ ]:
loc, rot, orientation = load_position("plate_pickup")
await arm.move_to_location(
  location=loc,
  direction=rot,
  backend_params=PreciseFlexBackend.MoveToLocationParams(orientation=orientation, speed=30),
)
print("Moved to 'plate_pickup'")

## Move with speed control

In [ ]:
await arm.move_to_location(
  location=Coordinate(300, 0, 200),
  direction=Rotation(0, 0, 0),
  backend_params=PreciseFlexBackend.MoveToLocationParams(speed=20),
)
print(await arm.request_gripper_location())

## Lefty / Righty

Pass `ElbowOrientation` through `backend_params`.

In [ ]:
await arm.move_to_location(
  location=Coordinate(300, 0, 200),
  direction=Rotation(0, 0, 0),
  backend_params=PreciseFlexBackend.MoveToLocationParams(
    orientation=ElbowOrientation.RIGHT,
    speed=30,
  ),
)
print("Righty:", await arm.request_gripper_location())

In [ ]:
await arm.move_to_location(
  location=Coordinate(300, 0, 200),
  direction=Rotation(0, 0, 0),
  backend_params=PreciseFlexBackend.MoveToLocationParams(
    orientation=ElbowOrientation.LEFT,
    speed=30,
  ),
)
print("Lefty:", await arm.request_gripper_location())

In [ ]:
await backend.change_config()
print("Flipped:", await arm.request_gripper_location())

## Move joints

In [ ]:
await arm.move_to_joint_position(
  {PFAxis.BASE: 45.0},
  backend_params=PreciseFlexBackend.MoveToJointPositionParams(speed=30),
)
print(await arm.request_joint_position())

## Gripper

In [ ]:
await arm.open_gripper(gripper_width=80.0)
print(f"Gripper closed: {await arm.is_gripper_closed()}")

In [ ]:
await arm.close_gripper(gripper_width=10.0)
print(f"Gripper closed: {await arm.is_gripper_closed()}")

## Disconnect

In [ ]:
await backend.move_to_safe()
await arm.stop()
print("Done")